# Annotation des taxonomies avec une IA

Ce notebook parcourt les images du dataset local et utilise Qwen3-VL pour proposer des étiquettes à partir des arbres de [`taxonomies.json`](taxonomies.json). Chaque dimension possède une hiérarchie : les feuilles deviennent des étiquettes et leur chemin complet est conservé. Chaque résultat est sauvegardé immédiatement ; relance le traitement pour reprendre après une interruption. Les annotations sont automatiques et demandent une validation humaine.

**Parcours :** configurer les chemins, charger le modèle, annoter, inspecter, puis exécuter la cellule de synchronisation si souhaité. Le modèle et le traitement peuvent demander beaucoup de temps et de mémoire GPU.


## Exemple de résultat JSON

Une ligne JSONL correspond à une image. `labels` contient les feuilles choisies par l’IA et `taxonomy_paths` contient leurs chemins complets, reconstruits et vérifiés à partir de `taxonomies.json`.

```json
{
  "source": {"root": "data", "relative_path": "jeu/personnage.png"},
  "labels": {"asset_type": "single_sprite", "subject_type": "character", "view": "front", "pose": "standing", "action": "idle", "background_type": "transparent", "render_family": "pixel_art"},
  "taxonomy_paths": {"asset_type": ["sprite", "single_sprite"], "subject_type": ["living", "character"], "view": ["horizontal", "front"], "pose": ["grounded", "upright", "standing"], "action": ["stationary", "idle"], "background_type": ["transparent"], "render_family": ["pixel_art"]},
  "caption_candidate": "A front-facing pixel-art character standing still.",
  "review_status": "auto_labeled",
  "needs_taxonomy_review": true
}
```


## 1. Dépendances

Utilise le kernel Python du projet avec PyTorch CUDA, puis installe les dépendances si nécessaire.


In [ ]:
%pip install -q "transformers>=4.57" accelerate huggingface_hub pillow tqdm


## 2. Configuration et fichiers

Le dataset local provient du téléchargement du viewer. `MAX_IMAGES = None` traite toutes les images ; commence par un petit nombre pour vérifier les résultats. Le jeton Hugging Face est lu depuis `HF_TOKEN` ou la connexion `hf auth login` ; il doit avoir accès en écriture pour la synchronisation.


In [ ]:
import json, os, hashlib
from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm
from PIL import Image

cwd = Path.cwd().resolve()
DATASET_NOTEBOOK = next((p for p in (cwd, cwd / "notebook/dataset", cwd / "dataset") if (p / "taxonomies.json").is_file()), None)
if DATASET_NOTEBOOK is None:
    raise FileNotFoundError("Ouvre le notebook depuis notebook/dataset, notebook ou la racine du projet.")
PROJECT_ROOT = DATASET_NOTEBOOK.parent.parent
DATASET_ROOT = PROJECT_ROOT / "dataset-viewer/datasets/pixelart-dataset/.gitrepo"
TAXONOMY_PATH = DATASET_NOTEBOOK / "taxonomies.json"
OUTPUT_PATH = DATASET_NOTEBOOK / "taxonomy_work/annotations_v2.jsonl"
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
REPO_ID = "ethanzxv/pixelart-dataset"
MAX_IMAGES = 10  # Mettre None pour traiter tout le dataset.

if not (DATASET_ROOT / "data").is_dir():
    raise FileNotFoundError(f"Dataset local introuvable : {DATASET_ROOT / 'data'}. Télécharge-le depuis le viewer.")
taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding="utf-8"))
def build_leaf_paths(tree, prefix=()):
    if not isinstance(tree, dict) or not tree:
        raise ValueError("Chaque hiérarchie doit être un arbre non vide.")
    paths = {}
    for name, children in tree.items():
        if not isinstance(name, str) or not name or not isinstance(children, dict):
            raise ValueError("Les nœuds doivent être des noms non vides associés à des objets.")
        path = (*prefix, name)
        if children:
            nested = build_leaf_paths(children, path)
            if paths.keys() & nested.keys():
                raise ValueError("Une feuille doit être unique dans sa dimension.")
            paths.update(nested)
        else:
            if name in paths:
                raise ValueError(f"Feuille dupliquée : {name}")
            paths[name] = list(path)
    return paths

hierarchies = taxonomy["hierarchies"]
leaf_paths = {dimension: build_leaf_paths(tree) for dimension, tree in hierarchies.items()}
fields = {dimension: list(paths) for dimension, paths in leaf_paths.items()}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
images = sorted(p for p in (DATASET_ROOT / "data").rglob("*") if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".gif"})
print(f"{len(images)} images trouvées. Résultats : {OUTPUT_PATH}")
print("Champs :", ", ".join(fields))


## 3. Charger le modèle visuel

Qwen analyse les pixels et ne déduit pas les droits ni le rôle de jeu. Un GPU NVIDIA est recommandé.


In [ ]:
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError("Active un kernel avec GPU CUDA avant de charger le modèle.")
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=224 * 224, max_pixels=512 * 512)
model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, device_map="auto", dtype="auto", attn_implementation="sdpa")
model.eval()
print("Modèle prêt :", MODEL_ID)


## 4. Annoter et reprendre

Le notebook n’écrase pas les annotations existantes du dataset. Les lignes déjà présentes dans le fichier de travail sont conservées ; seules les images manquantes sont traitées. Les erreurs sont affichées et réessayées à la prochaine exécution.


In [ ]:
def read_records(path):
    records = {}
    if path.exists():
        with path.open(encoding="utf-8") as f:
            for number, line in enumerate(f, 1):
                try:
                    row = json.loads(line)
                    key = row["source"]["relative_path"]
                except (ValueError, KeyError, TypeError) as exc:
                    raise ValueError(f"Ligne {number} invalide dans {path}; corrige-la avant de reprendre.") from exc
                if key in records:
                    raise ValueError(f"Doublon dans les résultats : {key}")
                records[key] = row
    return records

def validate_response(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    obj = json.loads(raw)
    labels = obj["labels"]
    if not isinstance(labels, dict) or set(labels) != set(fields):
        raise ValueError("Les champs de labels ne correspondent pas à la taxonomie.")
    for key, choices in fields.items():
        if labels[key] not in choices:
            raise ValueError(f"Valeur invalide pour {key}: {labels[key]}")
    caption = obj.get("caption_candidate")
    if not isinstance(caption, str) or not caption.strip():
        raise ValueError("Description absente.")
    return labels, caption.strip()

def annotate(path):
    instruction = ("Analyze only visible pixels. Return ONLY a JSON object with keys labels and caption_candidate. "
                   "labels must contain exactly these dimensions; choose one leaf name (an entry with an empty object) per dimension from this hierarchy: "
                   + json.dumps(hierarchies, ensure_ascii=False) +
                   ". Use unknown when unsure, not_applicable when irrelevant. Caption in English, concise and visual; do not infer game roles or rights.")
    with Image.open(path) as image:
        image = image.convert("RGBA")
        has_transparency = image.getchannel("A").getextrema()[0] < 255
        instruction += " The original image contains transparent pixels." if has_transparency else " The original image has no transparent pixels."
        # Nearest-neighbor enlargement makes tiny sprites legible without smoothing pixels.
        scale = max(1, min(8, 512 // max(image.size)))
        image = image.resize((image.width * scale, image.height * scale), Image.Resampling.NEAREST)
        background = Image.new("RGBA", image.size, "white")
        background.alpha_composite(image)
        messages = [{"role": "user", "content": [{"type": "image", "image": background.convert("RGB")}, {"type": "text", "text": instruction}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
        inputs.pop("token_type_ids", None)
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        text = processor.batch_decode(output[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return validate_response(text)

records = read_records(OUTPUT_PATH)
taxonomy_hash = hashlib.sha256(TAXONOMY_PATH.read_bytes()).hexdigest()
if any(row.get("taxonomy_sha256") != taxonomy_hash for row in records.values()):
    raise ValueError("La taxonomie a changé depuis ce lot ; choisis un nouveau OUTPUT_PATH pour éviter de mélanger les versions.")
pending = [p for p in images if p.relative_to(DATASET_ROOT / "data").as_posix() not in records]
if MAX_IMAGES is not None:
    pending = pending[:MAX_IMAGES]
errors = []
for path in tqdm(pending, desc="Annotation", unit="image"):
    relative = path.relative_to(DATASET_ROOT / "data").as_posix()
    try:
        labels, caption = annotate(path)
        row = {"asset_id": hashlib.sha256(("data/" + relative).encode()).hexdigest()[:24],
               "source": {"root": "data", "relative_path": relative}, "labels": labels,
               "taxonomy_paths": {key: leaf_paths[key][value] for key, value in labels.items()},
               "caption_candidate": caption, "caption_status": "candidate_unreviewed",
               "review_status": "auto_labeled", "needs_taxonomy_review": True,
               "rights_status": "unknown", "model": MODEL_ID, "taxonomy_version": taxonomy["version"], "taxonomy_sha256": taxonomy_hash,
               "annotated_at_utc": datetime.now(timezone.utc).isoformat()}
        with OUTPUT_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        records[relative] = row
    except Exception as exc:
        errors.append((relative, str(exc)))
print(f"{len(records)} annotations sauvegardées ; {len(errors)} erreurs ; {len(images) - len(records)} images restantes.")
for path, error in errors[:10]:
    print(f"- {path}: {error}")


## 5. Vérifier un exemple

Les étiquettes restent des propositions. Vérifie les images et les droits avant toute utilisation pour l'entraînement.


In [ ]:
from IPython.display import display
records = read_records(OUTPUT_PATH)
if records:
    example = next(iter(records.values()))
    display(Image.open(DATASET_ROOT / example["source"]["root"] / example["source"]["relative_path"]))
    print(json.dumps(example, indent=2, ensure_ascii=False))
else:
    print("Aucune annotation disponible ; exécute la cellule précédente.")


## 6. Synchroniser sur Hugging Face

Exécute cette cellule quand tu veux publier l'état actuel. Elle envoie le JSONL et la taxonomie ensemble dans un nouveau dossier horodaté sous `annotations/taxonomy_runs/` ; elle ne supprime ni ne remplace les fichiers existants. Les annotations sont marquées comme non vérifiées. Le jeton doit disposer des droits d'écriture sur le dataset.


In [ ]:
from huggingface_hub import HfApi, CommitOperationAdd

records = read_records(OUTPUT_PATH)
if not records:
    raise ValueError("Aucune annotation à synchroniser.")
current_hash = hashlib.sha256(TAXONOMY_PATH.read_bytes()).hexdigest()
if any(row.get("taxonomy_sha256") != current_hash for row in records.values()):
    raise ValueError("La taxonomie ne correspond pas aux annotations : utilise le fichier de taxonomie de ce lot.")
api = HfApi()
revision = api.repo_info(REPO_ID, repo_type="dataset").sha
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
remote_dir = f"annotations/taxonomy_runs/{run_name}"
commit = api.create_commit(
    repo_id=REPO_ID, repo_type="dataset", parent_commit=revision,
    operations=[CommitOperationAdd(path_in_repo=f"{remote_dir}/annotations.jsonl", path_or_fileobj=str(OUTPUT_PATH)),
                CommitOperationAdd(path_in_repo=f"{remote_dir}/taxonomies.json", path_or_fileobj=str(TAXONOMY_PATH))],
    commit_message=f"Add taxonomy annotations ({len(records)} images)",
)
print(f"{len(records)} annotations synchronisées dans {REPO_ID}/{remote_dir}")
print(commit.commit_url)
